# Project 3

group:
- Ole Lie-Bjelland
- Anniken Marie Lingstad
- Emil Ingar Hals

We did NOT use any AI for this project.

## Topic 1

### Task 0

Our coordnate box will use angstrom units. To ensure that all the points in the dna molecule fit in the box we found the lowest and highest of each x,y,z value in the dna_coords.txt file. The box is then expanded by 5 angstrom in every direction to ensure that the atomic radius of each point remains inside the simulation box

In [ ]:
from main import gen_simulation_box

simulation_box = gen_simulation_box(10, 10, 10)
simulation_box_volume = simulation_box.volume()

print(f"Simulation box: {simulation_box}")
print(f"Simulation box volume: {simulation_box_volume}")

### Task 1

In [ ]:
from main import gen_random_points

point_count = 3
points = gen_random_points(simulation_box, point_count)
print(f"Generated {len(points)} random points:")
for p in points:
    print(p)

### Task 2

In [ ]:
from main import gen_spheres

spheres = gen_spheres(simulation_box, 1)
for s in spheres:
    print(s)

### Task 3

In [ ]:
from main import point_in_spheres

rand_point = gen_random_points(simulation_box, 1)[0]
print(f"Point: \n{rand_point}")

assert point_in_spheres(spheres, spheres[0].point), "point_in_spheres function not working"   # This should always be true

if point_in_spheres(spheres, rand_point):
    print("The point is in the sphere")
else:
    print("The point is not in the sphere")


### Task 4

We can check the montecarlo simulation result by comparing it to the volume of a sphere given by the formula: $V = \frac{4}{3} \pi r^3$


In [ ]:
num_of_points = 500000
sphere = gen_spheres(simulation_box, 1)
random_points = gen_random_points(simulation_box, num_of_points)

print(f"sphere radius is: {sphere[0].rad}")

points_in_sphere = 0
for point in random_points:
    if point_in_spheres(sphere, point):
        points_in_sphere += 1

estimated_volume = simulation_box.volume() * (points_in_sphere/num_of_points)

print(f"The fraction of points in the sphere is {points_in_sphere/num_of_points}")
print(f"The volume of the simulation box is: {simulation_box.volume()}")
print(f"Estimated volume of sphere: {estimated_volume}")
print(f"Analytical volume of sphere: {sphere[0].volume()}")

# Plotting
from main import plot_points_and_spheres
plot_points_and_spheres(sphere, random_points[:250])   # Only plotting 250 points because plotting all of them is very slow


### Task 5

$$
V = \frac{4}{3} \pi r^3 \to \pi = \frac{3V}{4 \cdot r^3}
$$

In [ ]:
estimated_pi = (estimated_volume*3)/(4*sphere[0].rad**3)
print(f"Estimated pi: {estimated_pi}")

### Task 6

In [ ]:
spheres = gen_spheres(simulation_box, 10)

# Show the first 3 of the 10 new spheres
for s in spheres[:3]:
    print(s)

### Task 7

We can verify the results of the simulation in the same way as in task 4. As we know the radius of all the spheres we can calculate their total volume analytically.

In [ ]:
num_of_points = 300000
random_points = gen_random_points(simulation_box, num_of_points)

# Calculate the analytical volume
total_volume = 0
for s in spheres:
    total_volume += s.volume()

points_in_spheres = 0
for point in random_points:
    points_in_spheres += point_in_spheres(spheres, point)

fraction = points_in_spheres/num_of_points
estimated_volume = simulation_box.volume() * fraction
print(f"Total volume of simulation box: {simulation_box.volume()}")
print(f"The fraction of points in the sphere is {fraction}")
print(f"The total volume of the generated spheres is: {round(total_volume, 2)}")
print(f"The estimated volume is {estimated_volume}")

# Plotting
plot_points_and_spheres(spheres, random_points[:250])

### Task 8

In [ ]:
from main import get_atoms
atoms = get_atoms()
for a in atoms[:3]:
    print(a)

### Task 9

In [ ]:
from main import get_dna_box
dna_box = get_dna_box(atoms)
print("Simulation box for DNA:")
print(dna_box)
print(f"Volume of DNA box: {dna_box.volume()}")

### Task 10

In order to find the analytical volume of the DNA we can use the normal formula for the volume of a sphere as we already know the atomic radius of each atom.

In [ ]:
random_point_count = 100000
random_points = gen_random_points(dna_box, random_point_count)

# Check how many of the random points are in the dna
points_in_dna = 0
for p in random_points:
    points_in_dna += point_in_spheres(atoms, p)

fraction = points_in_dna/random_point_count
estimated_dna_volume = dna_box.volume()*fraction

print(f"Volume of simulation box: {dna_box.volume()}")
print(f"Fraction of points in dna: {fraction}")
print(f"Estimated dna volume: {estimated_dna_volume} angstrom^3")


# Analytical approach
import math
analytical_dna_volume = 0
for atom in atoms:
    analytical_dna_volume += math.pi * (4/3) * atom.rad**3

print(f"analytical dna volume: {analytical_dna_volume} angstrom^3")


## Topic 2

### Task 1

A simple random walk function without numpy to generate a set of walkers in 3d, starting from different random points. Imported from random_walk.py

In [ ]:
from main import random_walk

random_walk()

### Task 2

The idea is the same as for task 1 but we use numpy this time.

In [ ]:
from main import random_walk_fast

random_walk_fast()

### Task 3

We interpret "accessible volume of DNA" as being the surface area of the DNA structure. Our strategy to do this is to:

**First let the walkers walk**
- Divide the 3D space into blocks/cells
- Start a walker in a random block that is not inside an atom
- Let the walker move up, down, left, right, forwards or backwards
- if (new position is inside an atom): save the block and move the walker to a new random block and let it continue walking  
- else: keep walking
- repeat for all walkers


**Then interpret the collected data**
- Check how many uniue blocks were visited by the walkers (remove duplicates)
- Check how many unique surface blocks were discovered
- Calculate how many blocks are in the simulation box
- Calculate the ratio of surface block to non-surface block from the walker data.
- Scale to match the total ammount of points in the simulation box (estimated_surface_blocks = total_blocks * surface_block_fraction)

This approach should work with any shape assuming there is a way to check if any given point is indide the shape or not. More complex shapes would requires smaller block sizes, making this process slow for complex structures.

### Task 4

To verify that this strategy is correct we can test it with a simple shape that has a known surface area, like a sphere. We put the sphere in the simulation box and let the walkers walk round it, hopefully colliding with the sphere which would mark the block as a surface block. We then follow the rest of the steps described in task 3 and compare with the analytical surface area of the sphere.

If it works we can scale up the complexity by adding more spheres to better resemble the DNA structure, if doing this it is important that the test spheres dont overlap as the analytical approach does not take this into consideration.

We will be using this formula to calculate the exact surface area of the test spheres:
$$
A = 4\pi r^2
$$

### Task 5

To ensure that the spheres dont overlap we created an option to run the test with the same 3 spheres every time, to disable this set 'deterministic=False' in the function argument.

In [ ]:
from main import estimate_surface_area

estimate_surface_area(deterministic=True)

### Task 5

While working on the montecarlo simulation we discovered that the simulation has a relatively large standard deviation, but it also had a consistently accurate average estimate if running multiple simulations. After some experimentation we decided that taking the average of 5 simulations is a nice spot between speed and accuracy. 

In [14]:
from accessible_volume import accessible_volume_simulation

WALKER_COUNT = 250
STEPS = 100
results = []

# DNA simulation
atoms = get_atoms()
dna_box = get_dna_box(atoms)
for i in range(5):
    result, _ = accessible_volume_simulation(WALKER_COUNT, STEPS, dna_box, atoms)
    results.append(result)
average = sum(results) / len(results)

print("-"*40 + "\nDNA Simulation Result")
print(f"Average result over 5 runs: {average}")

Estimated volume of spheres: 2704.5516
Actual volume of spheres: 3672.2768725392457
Box volume: 26970
Estimated accessible volume: (box volume - estimated volume of spheres) = 24265.4484
Estimated volume of spheres: 3467.2632000000003
Actual volume of spheres: 3672.2768725392457
Box volume: 26970
Estimated accessible volume: (box volume - estimated volume of spheres) = 23502.7368
Estimated volume of spheres: 3276.3156
Actual volume of spheres: 3672.2768725392457
Box volume: 26970
Estimated accessible volume: (box volume - estimated volume of spheres) = 23693.6844
Estimated volume of spheres: 4050.894
Actual volume of spheres: 3672.2768725392457
Box volume: 26970
Estimated accessible volume: (box volume - estimated volume of spheres) = 22919.106
Estimated volume of spheres: 2816.7468000000003
Actual volume of spheres: 3672.2768725392457
Box volume: 26970
Estimated accessible volume: (box volume - estimated volume of spheres) = 24153.2532
----------------------------------------
DNA Simu